# Cào dữ liệu Phongtro123 cho Đồ án Data Science  

Notebook này triển khai **crawler đa luồng** để thu thập dữ liệu phòng trọ tại TP.HCM từ website `phongtro123.com`.

### Mục tiêu chính
- Thu thập dữ liệu: `url, title, price, area, address, description, posted_time, owner_name, phone, source`  
- Tối ưu tốc độ bằng `requests.Session` + `ThreadPoolExecutor`.  
- Hạn chế bị chặn bằng cách xoay vòng User-Agent, delay nhẹ, retry hợp lý.

Cấu trúc notebook được chia nhỏ theo từng khối logic, giúp dễ đọc, dễ debug như phong cách làm việc của một Data Scientist.


## 1. Import thư viện & cấu hình cơ bản

Ở bước này ta:
- Import các thư viện chuẩn (HTTP, HTML parsing, regex, đa luồng, thời gian…).
- Khai báo các hằng số cấu hình: domain, URL list, số luồng, delay, user-agents...


In [ ]:
import re
import csv
import time
import random
import json
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


# =========================================================
# CONFIG
# =========================================================
BASE_DOMAIN = "https://phongtro123.com"
BASE_LIST_URL = "https://phongtro123.com/tinh-thanh/ho-chi-minh"
SOURCE_NAME = "phongtro123"

# Số luồng cào trang chi tiết
MAX_WORKERS = 50  # có thể giảm nếu máy yếu hoặc sợ bị chặn

# Delay nhỏ giữa các request chi tiết (0 = tắt delay)
DETAIL_DELAY_RANGE = (0.05, 0.2)

# Số trang list liên tiếp không có dữ liệu trước khi dừng
EMPTY_STOP = 3

# Danh sách User-Agent để xoay vòng
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_6) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
]

## 2. Định nghĩa Regex

Phần này gom tất cả các mẫu Regex dùng để:
- Nhận diện link bài đăng (`-prxxxxx.html`).  
- Bắt số điện thoại, thời gian đăng, cụm giá, diện tích…  
- Hỗ trợ logic tìm giá/diện tích trong toàn bộ text trang.


In [ ]:
# =========================================================
# REGEX CHO LINK / PHONE / THỜI GIAN
# =========================================================
RE_POST_LINK = re.compile(r"-pr\d+\.html?$", re.I)
RE_PHONE = re.compile(r"\b0\d{8,10}\b")
RE_TIME_DATE = re.compile(r"(\d{1,2}:\d{2})\s*(\d{1,2}/\d{1,2}/\d{4})")
RE_NGAY_DANG = re.compile(r"ngày\s*đăng\s*:?\s*([^\n]+)", re.I)

# =========================================================
# REGEX CHO GIÁ / DIỆN TÍCH (gắn label hoặc xuất hiện trong text)
# =========================================================
RE_PRICE_LABEL = re.compile(
    r"Giá[^0-9]{0,20}([0-9]{1,3}(?:[.,][0-9]{1,3})*(?:[.,][0-9]{1,2})?)\s*(triệu|tr|nghìn|ngàn|k|đ|vnđ|vnd)",
    re.I
)
RE_PRICE_ANY = re.compile(
    r"([0-9]{1,3}(?:[.,][0-9]{1,3})*(?:[.,][0-9]{1,2})?)\s*(triệu|tr|nghìn|ngàn|k|đ|vnđ|vnd)"
    r"(?:\s*\/\s*tháng|\s*tháng|\s*\/\s*người|\s*người)?",
    re.I
)

RE_AREA_LABEL = re.compile(
    r"Diện tích[^0-9]{0,20}([0-9]{1,3}(?:[.,][0-9]{1,2})?)\s*(m2|m²|met vuong|m vuong)",
    re.I
)
RE_AREA_ANY = re.compile(
    r"([0-9]{1,3}(?:[.,][0-9]{1,2})?)\s*(m2|m²|met vuong|m vuong)",
    re.I
)

# Giá trên dòng đầu (màu xanh lá): 2.7 triệu/tháng ...
RE_PRICE_INLINE = re.compile(
    r"(\d[\d\.\,]*)\s*(triệu/tháng|triệu|đồng/tháng|đồng)",
    re.IGNORECASE,
)

# Diện tích: 16 m2, 16 m², 16 m 2, 16 m^2 ...
RE_AREA_INLINE = re.compile(
    r"(\d[\d\.\,]*)\s*(m2|m²|m vuông|met vuong|m vuong|m\s*\^\s*\{?2\}?|m\s*\^\s*2|m\s*2\b)",
    re.IGNORECASE,
)

## 3. Các hàm tiện ích chung (text, JSON, thời gian)

Nhóm hàm này giúp:
- Chuẩn hóa khoảng trắng.  
- Chuẩn hóa cách ghi đơn vị.  
- Đọc JSON trong các thẻ `<script>` an toàn.  
- Đổi định dạng datetime ISO sang định dạng `HH:MM dd/mm/yyyy`.


In [ ]:
# =========================================================
# HELPER: TEXT & JSON & DATETIME
# =========================================================
def normalize_space(s: str):
    """Loại bỏ khoảng trắng thừa, đảm bảo string gọn gàng."""
    return re.sub(r"\s+", " ", (s or "")).strip()

def fix_unit_spacing(s: str):
    """Chuẩn hóa cách viết đơn vị như m2, triệu/tháng, tr/tháng."""
    s = normalize_space(s)
    s = re.sub(r"\bm\s*2\b", "m2", s, flags=re.I)
    s = re.sub(r"\btriệu\s*\/\s*tháng\b", "triệu/tháng", s, flags=re.I)
    s = re.sub(r"\btr\s*\/\s*tháng\b", "tr/tháng", s, flags=re.I)
    return s

def pick_first_text(soup, selectors):
    """Lấy text của selector đầu tiên tìm được (nếu có)."""
    for sel in selectors:
        tag = soup.select_one(sel)
        if tag:
            txt = tag.get_text(" ", strip=True)
            if txt:
                return txt
    return ""

def extract_by_label_text(visible_text, labels):
    """Tìm dòng có label cụ thể (ví dụ: 'Địa chỉ', 'Khu vực') trong full-text."""
    for lb in labels:
        m1 = re.search(rf"{lb}\s*:\s*([^\n]+)", visible_text, flags=re.I)
        if m1:
            val = m1.group(1).strip()
            if val:
                return val
        m2 = re.search(rf"{lb}\s*:\s*\n\s*([^\n]+)", visible_text, flags=re.I)
        if m2:
            val = m2.group(1).strip()
            if val:
                return val
    return ""

def safe_json_load(s):
    """Đọc JSON từ chuỗi, nếu lỗi thì trả về None (không throw)."""
    try:
        return json.loads(s)
    except Exception:
        return None

def find_in_json(obj, keys):
    """Duyệt đệ quy trong JSON để tìm value của các key mong muốn."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys and isinstance(v, (str, int, float)):
                return str(v)
            found = find_in_json(v, keys)
            if found:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = find_in_json(it, keys)
            if found:
                return found
    return ""

def iso_to_hhmm_ddmmyyyy(s):
    """Đổi datetime ISO sang định dạng HH:MM dd/mm/yyyy nếu có thể."""
    try:
        s = s.strip().replace("Z", "+00:00")
        dt = datetime.fromisoformat(s)
        return dt.strftime("%H:%M %d/%m/%Y")
    except Exception:
        return 

## 4. Tầng HTTP: Session theo thread & hàm fetch()

Để tối ưu hiệu năng:
- Mỗi thread dùng một `requests.Session` riêng (thread-local).  
- `fetch()` có retry nhẹ, delay nhỏ giữa các lần retry.  
- Tự động gắn header với User-Agent ngẫu nhiên.


In [ ]:
# =========================================================
# THREAD-LOCAL SESSION (requests.Session không thread-safe)
# =========================================================
_thread_local = threading.local()

def get_session():
    """Mỗi thread có một session riêng để tái sử dụng kết nối HTTP."""
    if getattr(_thread_local, "session", None) is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=50,
            pool_maxsize=50,
            max_retries=0
        )
        s.mount("http://", adapter)
        s.mount("https://", adapter)
        _thread_local.session = s
    return _thread_local.session


# =========================================================
# HTTP
# =========================================================
def make_headers(referer=None):
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive",
        "Referer": referer or BASE_DOMAIN,
    }

def fetch(url, referer=None, retries=3, timeout=20):
    """
    Gửi request GET với session pool + retry nhẹ.
    Trả về HTML (str) nếu thành công, ngược lại trả về None.
    """
    session = get_session()
    last_err = None
    for i in range(retries):
        try:
            r = session.get(url, headers=make_headers(referer), timeout=timeout)
            if r.status_code == 200 and r.text.strip():
                low = r.text.lower()
                if "just a moment" in low or "enable javascript" in low:
                    last_err = "cloudflare page"
                else:
                    return r.text
            else:
                last_err = f"status={r.status_code}"
        except Exception as e:
            last_err = str(e)

        # retry delay ngắn
        time.sleep(0.6 + i * 0.6)
    print(f"[WARN] fetch failed {last_err} at {url}")
    return None

## 5. Lọc tin thuộc TP.HCM

Website có thể chứa nhiều tin ở các tỉnh/thành khác.  
Ta chỉ giữ lại bài đăng có liên quan tới:
- “Hồ Chí Minh”, “Ho Chi Minh”, “TP.HCM”, “TPHCM” trong địa chỉ hoặc text trang.


In [ ]:
# =========================================================
# HCM FILTER
# =========================================================
def is_hcm_address(address: str, extra_text: str = "") -> bool:
    """Kiểm tra địa chỉ có thuộc khu vực TP.HCM hay không."""
    low = normalize_space((address or "") + " " + (extra_text or "")).lower()
    return bool(re.search(r"\b(hồ\s*chí\s*minh|ho\s*chi\s*minh|tp\.?\s*hcm|tphcm)\b", low))

## 6. Trích xuất link bài đăng từ trang danh sách

Từ URL list dạng:  
`https://phongtro123.com/tinh-thanh/ho-chi-minh?page=N`

Ta:
- Dùng `BeautifulSoup` để tìm các block bài đăng.  
- Lọc ra các link có pattern `-prxxxxx.html`.  
- Loại bỏ các link thuộc `/tags/`, `/blog/`, `/tin-tuc/`.


In [ ]:
# =========================================================
# LIST PAGE -> LINKS
# =========================================================
def get_post_links_from_list(page_number):
    list_url = f"{BASE_LIST_URL}?page={page_number}"
    html = fetch(list_url, referer=BASE_LIST_URL)
    if not html:
        return [], list_url

    soup = BeautifulSoup(html, "html.parser")
    links = set()

    post_items = soup.select(
        "li.post-item, article.post-item, div.post-item, "
        "li.post-listing-item, article.post-listing-item, div.post-listing-item"
    )

    for item in post_items:
        a = item.select_one("h3 a[href], h2 a[href], a[href]")
        if not a:
            continue
        href = a.get("href", "")
        if href and RE_POST_LINK.search(href):
            full = urljoin(BASE_DOMAIN, href.split("?")[0])
            low = full.lower()
            if any(x in low for x in ["/tags/", "/blog/", "/tin-tuc/"]):
                continue
            links.add(full)

    # Fallback: nếu CSS selector không match, dùng regex thô trên HTML
    if not links:
        for m in re.finditer(r'href="([^"]*-pr\d+\.html?)"', html):
            href = m.group(1)
            full = urljoin(BASE_DOMAIN, href.split("?")[0])
            low = full.lower()
            if any(x in low for x in ["/tags/", "/blog/", "/tin-tuc/"]):
                continue
            links.add(full)

    return list(links), list_url

## 7. Trích xuất mô tả đầy đủ (description)

Phần mô tả có thể nằm trong nhiều block HTML khác nhau, nên ta:
- Ưu tiên các container chuẩn (`post-main-content`, `post-content`, `description`...).  
- Fallback: tìm heading có text giống “Thông tin mô tả” rồi gom các đoạn phía sau.


In [ ]:
# =========================================================
# DESCRIPTION
# =========================================================
def extract_full_description(soup):
    cont = soup.select_one(
        "div.post-main-content, section.post-main-content, "
        "div#post-content, div.post-content, "
        "div.post-description, section.post-description, "
        "div.description"
    )
    if cont:
        txt = cont.get_text("\n", strip=True)
        return normalize_space(txt.replace("\r", "\n"))

    heading = soup.find(
        lambda t: t.name in ["h2", "h3", "h4"]
        and "thông tin mô tả" in t.get_text(strip=True).lower()
    )
    if heading:
        texts = []
        node = heading
        while True:
            node = node.find_next_sibling()
            if not node:
                break
            cls = " ".join(node.get("class", [])).lower()
            if node.name in ["h2", "h3", "h4"] and "thông tin mô tả" not in node.get_text(strip=True).lower():
                break
            if any(k in cls for k in ["post-attributes", "box-user-info", "user-info", "section-contact", "post-summary"]):
                break
            t = node.get_text("\n", strip=True)
            if t:
                texts.append(t)
        if texts:
            return normalize_space("\n".join(texts))

    return 

## 8. Trích xuất thời gian đăng tin (`posted_time`)

Trang có thể hiển thị thời gian đăng ở nhiều chỗ:
- Dòng kiểu “Ngày đăng: …” trong block thuộc tính.  
- Thẻ `<time>` với `datetime`.  
- Meta tag `article:published_time`.  
- Hoặc các pattern `HH:MM dd/mm/yyyy` trong full-text.


In [ ]:
# =========================================================
# POSTED TIME
# =========================================================
def extract_posted_time(soup, visible_text):
    # 1. Ưu tiên dạng "Ngày đăng: ..."
    m = RE_NGAY_DANG.search(visible_text)
    if m:
        raw = m.group(1).strip()
        mm = RE_TIME_DATE.search(raw)
        if mm:
            return f"{mm.group(1)} {mm.group(2)}"
        return raw

    # 2. Tìm trong các dòng thuộc tính bài đăng
    for row in soup.select(
        ".post-attributes li, .post-attributes .item, "
        ".post-summary .summary-item, .summary-item"
    ):
        t = row.get_text(" ", strip=True)
        if t and re.search(r"ngày\s*đăng", t, re.I):
            mm = RE_TIME_DATE.search(t)
            if mm:
                return f"{mm.group(1)} {mm.group(2)}"
            t2 = re.sub(r"ngày\s*đăng\s*:?\s*", "", t, flags=re.I).strip()
            if t2:
                return t2

    # 3. Thẻ <time>
    time_tag = soup.find("time")
    if time_tag:
        raw = time_tag.get_text(" ", strip=True)
        if raw and not re.search(r"cập\s*nhật", raw, re.I):
            mm = RE_TIME_DATE.search(raw)
            if mm:
                return f"{mm.group(1)} {mm.group(2)}"
            dt_attr = time_tag.get("datetime")
            if dt_attr:
                formatted = iso_to_hhmm_ddmmyyyy(dt_attr)
                if formatted:
                    return formatted
            return raw

    # 4. Meta article:published_time
    meta = soup.select_one("meta[property='article:published_time']")
    if meta and meta.get("content"):
        formatted = iso_to_hhmm_ddmmyyyy(meta["content"])
        if formatted:
            return formatted

    # 5. Fallback: quét pattern HH:MM dd/mm/yyyy trong full-text
    matches = list(RE_TIME_DATE.finditer(visible_text))
    if matches:
        for mt in matches:
            start = max(0, mt.start() - 30)
            window = visible_text[start:mt.start()].lower()
            if "cập nhật" not in window:
                return f"{mt.group(1)} {mt.group(2)}"
        first = matches[0]
        return f"{first.group(1)} {first.group(2)}"

    return 

## 9. Trích xuất số điện thoại (`phone`)

- Ưu tiên đọc từ link `tel:` trong HTML.  
- Nếu không có, dùng Regex bắt số điện thoại trong full-text.


In [ ]:
# =========================================================
# PHONE
# =========================================================
def extract_phone(soup, visible_text):
    tel = soup.select_one('a[href^="tel:"]')
    if tel:
        num = tel.get("href", "").replace("tel:", "").strip()
        if num:
            return re.sub(r"\D", "", num)
    m = RE_PHONE.search(visible_text.replace(" ", ""))
    return m.group(0)

## 10. Hàm hỗ trợ chuẩn hóa giá & diện tích

- Đảm bảo `price` có đơn vị gắn với tháng/người nếu cần.  
- Đảm bảo `area` có đơn vị m2.  
Ngoài ra còn có **logic cũ (legacy)** để fallback khi logic mới không bắt được.


In [ ]:
# =========================================================
# PRICE / AREA – CHUẨN HÓA ĐƠN VỊ
# =========================================================
def ensure_month_unit(price_text: str):
    if not price_text:
        return ""
    low = price_text.lower()
    if any(x in low for x in ["tháng", "/thang", "người", "/nguoi", "m2", "m²"]):
        return price_text
    return price_text + "/tháng"

def ensure_m2_unit(area_text: str):
    if not area_text:
        return ""
    low = area_text.lower()
    if "m2" in low or "m²" in low or "vuong" in low:
        return area_text
    if re.fullmatch(r"[0-9.,]+", area_text.strip()):
        return area_text.strip() + " m2"
    return area_text


# =========================================================
# LOGIC CŨ (LEGACY) – FALLBACK CHO GIÁ & DIỆN TÍCH
# =========================================================
def _extract_price_legacy(soup, visible_text, title="", desc=""):
    price = pick_first_text(soup, [
        "span.item-price",
        ".post-summary .summary-item .price",
        ".post-attributes .price",
        "span.price",
        "div.price"
    ])
    if price:
        return ensure_month_unit(fix_unit_spacing(price))

    for sc in soup.select('script[type="application/ld+json"]'):
        data = safe_json_load(sc.string or sc.get_text())
        if data:
            v = find_in_json(data, keys={"price", "lowPrice", "highPrice"})
            if v:
                return ensure_month_unit(fix_unit_spacing(v))

    m = RE_PRICE_LABEL.search(visible_text) or RE_PRICE_ANY.search(visible_text)
    if m:
        return ensure_month_unit(fix_unit_spacing(f"{m.group(1)} {m.group(2)}"))

    mix = f"{title}\n{desc}"
    m = RE_PRICE_LABEL.search(mix) or RE_PRICE_ANY.search(mix)
    if m:
        return ensure_month_unit(fix_unit_spacing(f"{m.group(1)} {m.group(2)}"))

    return ""

def _extract_area_legacy(soup, visible_text, title="", desc=""):
    area = pick_first_text(soup, [
        "span.item-area",
        ".post-summary .summary-item .acreage",
        ".post-attributes .acreage",
        "span.acreage",
        "div.acreage"
    ])
    if area:
        return ensure_m2_unit(fix_unit_spacing(area))

    m = RE_AREA_LABEL.search(visible_text) or RE_AREA_ANY.search(visible_text)
    if m:
        return ensure_m2_unit(fix_unit_spacing(f"{m.group(1)} m2"))

    mix = f"{title}\n{desc}"
    m = RE_AREA_LABEL.search(mix) or RE_AREA_ANY.search(mix)
    if m:
        return ensure_m2_unit(fix_unit_spacing(f"{m.group(1)} m2"))

    return 

## 11. Logic mới: bắt giá & diện tích gần nhau trong full-text

Ý tưởng:
- Lấy toàn bộ text trang về một chuỗi dài.  
- Tìm **giá đầu tiên** xuất hiện (nhiều khả năng là giá chính, in đậm).  
- Từ vị trí đó, tìm **diện tích gần nhất phía sau**, trong một khoảng ký tự giới hạn.  
- Nếu không tìm được theo logic mới → fallback sang logic cũ.


In [ ]:
# =========================================================
# LOGIC MỚI: GIÁ & DIỆN TÍCH GẦN NHAU TRONG TEXT
# =========================================================
def find_price_area_from_text_basic(soup):
    """
    - Lấy toàn bộ text trang thành 1 chuỗi.
    - Tìm GIÁ đầu tiên (RE_PRICE_INLINE).
    - Từ vị trí đó, tìm DIỆN TÍCH gần nhất phía sau (RE_AREA_INLINE, trong 200 ký tự).
    Trả về: (context, price, area)
    """
    full = soup.get_text(" ", strip=True)
    if not full:
        return "", "", ""

    # 1. Giá đầu tiên
    m_price = RE_PRICE_INLINE.search(full)
    if not m_price:
        return "", "", ""

    raw_price = f"{m_price.group(1)} {m_price.group(2)}"
    price = ensure_month_unit(fix_unit_spacing(raw_price))
    price_end = m_price.end()

    # 2. Diện tích gần nhất phía sau
    best_area_match = None
    best_delta = None

    for m_area in RE_AREA_INLINE.finditer(full):
        if m_area.start() < price_end:
            continue
        delta = m_area.start() - price_end
        if delta > 200:  # tránh nhặt m2 ở quá xa trong mô tả
            break
        if best_area_match is None or delta < best_delta:
            best_area_match = m_area
            best_delta = delta

    area = ""
    if best_area_match is not None:
        raw_area = f"{best_area_match.group(1)} {best_area_match.group(2)}"
        area = ensure_m2_unit(fix_unit_spacing(raw_area))

    # 3. Context để debug (nếu cần)
    ctx_start = max(0, m_price.start() - 50)
    ctx_end = min(len(full), (best_area_match.end() if best_area_match else m_price.end()) + 50)
    context = full[ctx_start:ctx_end]

    return context, price, area


def extract_price(soup, visible_text, title="", desc=""):
    """Ưu tiên logic mới, nếu fail thì fallback logic cũ."""
    _, price, _ = find_price_area_from_text_basic(soup)
    if price:
        return price
    return _extract_price_legacy(soup, visible_text, title=title, desc=desc)


def extract_area(soup, visible_text, title="", desc=""):
    """Ưu tiên logic mới, nếu fail thì fallback logic cũ."""
    _, _, area = find_price_area_from_text_basic(soup)
    if area:
        return area
    return _extract_area_legacy(soup, visible_text, title=title, desc=desc)

## 12. Làm sạch & trích xuất tên chủ tin (`owner_name`)

- Bỏ các từ rác như “zalo, điện thoại, liên hệ…”.  
- Xóa chuỗi số dài và số điện thoại lẫn trong tên.  
- Lần lượt thử nhiều selector khác nhau, sau đó fallback sang scan theo heading “Thông tin liên hệ”.


In [ ]:
# =========================================================
# OWNER NAME
# =========================================================
RE_OWNER_JUNK = re.compile(
    r"(zalo|mobi|sđt|điện thoại|liên hệ|nhắn|tin đăng|tham gia|đang hoạt động)",
    re.I
)

def clean_owner_name(raw: str):
    if not raw:
        return ""
    raw = re.sub(r"\d[\d\.\s]{6,}\d", " ", raw)
    raw = re.sub(r"\b0\d{8,10}\b", " ", raw)
    raw = RE_OWNER_JUNK.sub(" ", raw)
    raw = normalize_space(raw)
    raw = re.sub(r"\d+", " ", raw)
    return normalize_space(raw)

def extract_owner_name(soup, page_text):
    candidates = []
    selectors = [
        ".post-contact .contact-name",
        ".post-contact .name",
        ".box-user-info .name",
        ".box-user-info .user-name",
        ".contact-info .name",
        ".user-info .name",
        ".user-info .user-name",
        ".post-author .author-name",
        ".author-name",
        "div.user-info strong",
        "div.user-info h3",
        "div.user-info h4",
    ]
    for sel in selectors:
        for tag in soup.select(sel):
            txt = clean_owner_name(tag.get_text(" ", strip=True))
            if txt and len(txt) >= 2:
                candidates.append(txt)

    if candidates:
        # Ưu tiên tên ngắn nhất (thường là tên thật)
        return sorted(set(candidates), key=len)[0]

    # Fallback: tìm theo heading "Thông tin liên hệ"
    heading = soup.find(lambda t: t.name in ["h2","h3","h4"] and "Thông tin liên hệ" in t.get_text())
    if heading:
        section = heading.find_parent(["section","div","aside"]) or heading.parent
        if section:
            for s in section.stripped_strings:
                s = normalize_space(s)
                if not s or RE_PHONE.search(s) or RE_OWNER_JUNK.search(s):
                    continue
                s2 = clean_owner_name(s)
                if s2 and len(s2) >= 2:
                    return s2

    # Fallback cuối: scan line sau dòng 'Thông tin liên hệ' trong full-text
    lines = [normalize_space(x) for x in page_text.split("\n") if normalize_space(x)]
    for i, line in enumerate(lines):
        if "Thông tin liên hệ" in line:
            for j in range(i+1, min(i+8, len(lines))):
                cand = clean_owner_name(lines[j])
                if cand and len(cand) >= 2:
                    return cand

    return 

## 13. Hàm parse chi tiết 1 bài đăng (`parse_detail`)

Quy trình:
1. Gửi request tới `post_url`.  
2. Dùng các hàm phía trên để trích xuất: `title, address, description, price, area, posted_time, owner_name, phone`.  
3. Lọc bỏ bài không thuộc TP.HCM.  
4. Bỏ bài nếu không có `title` hoặc cả `price` lẫn `area` đều trống.


In [ ]:
# =========================================================
# DETAIL PARSER (CHẠY TRONG LUỒNG)
# =========================================================
def parse_detail(post_url, referer):
    html = fetch(post_url, referer=referer)
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")
    visible_text = soup.get_text("\n", strip=True)

    title = pick_first_text(soup, [
        "h1.page-h1", "h1.post-title-lg", "h1.post-title", "h1"
    ])

    address = pick_first_text(soup, [
        "address.post-address",
        "div.post-address",
        "span.post-address",
        ".post-summary address",
        ".summary-item.address",
        "address"
    ]) or extract_by_label_text(visible_text, ["Địa chỉ", "Khu vực"])

    # Lọc TP.HCM
    if not is_hcm_address(address, visible_text):
        return None

    description = extract_full_description(soup)
    price = extract_price(soup, visible_text, title=title, desc=description)
    area  = extract_area(soup, visible_text, title=title, desc=description)

    posted_time = extract_posted_time(soup, visible_text)
    owner_name = extract_owner_name(soup, visible_text)
    phone = extract_phone(soup, visible_text)

    if not title:
        return None
    if (not price) and (not area):
        return None

    # Delay nhỏ giữa các request chi tiết
    if DETAIL_DELAY_RANGE and DETAIL_DELAY_RANGE != (0, 0):
        time.sleep(random.uniform(*DETAIL_DELAY_RANGE))

    return {
        "url": post_url,
        "title": title,
        "price": price,
        "area": area,
        "address": address,
        "description": description,
        "posted_time": posted_time,
        "owner_name": owner_name,
        "phone": phone,
        "source": SOURCE_NAME
    }

## 14. Crawler chính: duyệt list page + cào chi tiết song song

Hàm `crawl_phongtro123` sẽ:
- Duyệt tuần tự qua các trang list (page 1, 2, 3,…).  
- Trích xuất link bài đăng, loại bỏ URL trùng.  
- Dùng `ThreadPoolExecutor` để gọi `parse_detail` song song.  
- Ghi dữ liệu dòng nào xong dòng đó vào file CSV.


In [ ]:
# =========================================================
# MAIN CRAWLER (LIST SEQ + DETAIL PARALLEL)
# =========================================================
def crawl_phongtro123(start_page=1, end_page=None, output_csv="phongtro123.csv",
                      target_rows=None, empty_stop=EMPTY_STOP):

    fieldnames = [
        "url","title","price","area",
        "address","description",
        "posted_time","owner_name","phone","source"
    ]

    seen = set()
    written = 0
    empty_streak = 0
    page = start_page

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

        # Executor sống xuyên suốt để reuse thread + session pool
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            while True:
                if end_page is not None and page > end_page:
                    print("Reached end_page. Stop.")
                    break

                links, list_url = get_post_links_from_list(page)
                print(f"[LIST] Page {page}: {list_url}")
                print(f"  -> found {len(links)} post links")

                if not links:
                    empty_streak += 1
                    if end_page is None and empty_streak >= empty_stop:
                        print("No more pages. Stop.")
                        break
                    page += 1
                    continue
                empty_streak = 0

                # Lọc trùng trước khi submit
                new_links = []
                for link in links:
                    if link not in seen:
                        seen.add(link)
                        new_links.append(link)

                if not new_links:
                    page += 1
                    continue

                # Submit đa luồng cào detail
                futures = [ex.submit(parse_detail, link, list_url) for link in new_links]

                for fut in as_completed(futures):
                    item = None
                    try:
                        item = fut.result()
                    except Exception as e:
                        print(f"    [ERR] detail exception: {e}")

                    if item:
                        w.writerow(item)
                        written += 1
                        print(
                            f"    OK: {item['posted_time']} | {item['address']} | "
                            f"{item['price']} | {item['area']} | {item['owner_name']}"
                        )
                    
                    if target_rows is not None and written >= target_rows:
                        print(f"Reached target_rows={target_rows}. Stop.")
                        return

                page += 1

    print(f"Done. rows written={written}, unique urls={len(seen)} -> {output_csv}")

## 15. Cách chạy crawler (chỉ bật khi thật sự muốn cào)

Gợi ý 2 cách dùng:

### 15.1. Cào cho tới khi hết trang
```python
crawl_phongtro123(start_page=1, output_csv="phongtro123.csv")
```

### 15.2. Cào tới khi đủ `target_rows`
```python
crawl_phongtro123(start_page=1, target_rows=12000, output_csv="phongtro123_raw.csv")
```

> Khuyến nghị: test trước với `target_rows` nhỏ (ví dụ 50–100 dòng) để kiểm tra đúng cấu trúc dữ liệu và xem website có chặn hay không.


In [ ]:
# Chỉ chạy crawler khi bạn thật sự muốn cào dữ liệu.
if __name__ == "__main__":
    # Ví dụ 1: cào đến khi hết trang
    crawl_phongtro123(start_page=1, output_csv="../../data/raw/phongtro123_raw.csv")
    
    # Ví dụ 2: cào đến khi đủ 12.000 dòng
    # crawl_phongtro123(start_page=1, target_rows=12000, output_csv="phongtro123_raw.csv")
    pass